In [ ]:
import wandb
wandb.login()

# Load dependencies

In [ ]:
import os, random, logging
from pathlib import Path
from dataclasses import dataclass, field, asdict

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from transformers import ViTForImageClassification, ViTImageProcessor
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Using device: cuda
GPU: Tesla T4


# Helper functions

In [ ]:
import zipfile
def unzip(zip_path, extract_to):
  """Unzip folders"""
  with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

In [ ]:
def get_symdiff(set1, set2):
  """Get the symmetric difference of 2 sets"""
  return set1.symmetric_difference(set2)

# Load raw data


*   Just use train_data. Because test_data_v2 doesn't have labels.
*   Load train_data.zip to /content/ each time for fast access.



In [ ]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define paths
folder_path = "/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/"
drive_zip_path = folder_path + "train_data.zip"
local_zip_path = "/content/train_data.zip"
local_extract_path = "/content/train_data"

DATA_ROOT = Path('/content/train_data')
CSV_PATH  = folder_path + 'train.csv'

In [ ]:
import os
import shutil

# 1. Copy zip from Drive to local (fast access)
if os.path.exists(drive_zip_path):
    print(f"Copying '{drive_zip_path}' to '{local_zip_path}'...")
    shutil.copy(drive_zip_path, local_zip_path)
    print("Copy completed.")
else:
    print(f"Error: File not found at {drive_zip_path}")

# 2. Unzip locally
if os.path.exists(local_zip_path):
    print(f"Unzipping to '{local_extract_path}'...")
    # Ensure the destination directory exists
    os.makedirs(local_extract_path, exist_ok=True)
    unzip(local_zip_path, local_extract_path)
    print("Unzip completed.")

    # Check content
    print(f"Content of {local_extract_path}:", os.listdir(local_extract_path))

    # Set the new root for image loading
    # Since your csv contains paths like 'train_data/xxx.jpg', we point to the parent of 'train_data'
    image_root_path = local_extract_path
    print(f"\nSet 'image_root_path' to: {image_root_path}")
else:
    print("Zip file copy failed, cannot unzip.")

Copying '/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/train_data.zip' to '/content/train_data.zip'...
Copy completed.
Unzipping to '/content/train_data'...
Unzip completed.
Content of /content/train_data: ['485a102c1dc042dc91a03510b778807c.jpg', '9bb37eb79e95418da4c5f7cac80888eb.jpg', '5b63b54060d143a4ad37b43bf4cb5363.jpg', '4765a34ff50f43c8b5edff0676d27419.jpg', '00314cbc0d4248cbafd13d327f9198f5.jpg', '5957fe9a7d054d3da05c822036ee2e53.jpg', 'd71240a01b294e4ab6e698cfa302f7dc.jpg', '764c6000119d45f7896df3c66f188695.jpg', 'e188a53fc3784167844993e7097cc646.jpg', '4fb43b38092d4fa98099ab657d210ebf.jpg', '6902dc5cb6384a6ea96f45c1c7f34194.jpg', '6e68beef42654a2bb3e6e3432e21edc3.jpg', '2d645811a8b54a2c837e72db36feaeb8.jpg', '49a0fb7d782743828b24171296cd3a33.jpg', '21150b3da6454184b1cc560e7431ead3.jpg', '9e16cfbf5c8d4b8e82176087bd283365.jpg', 'ef5efdbba65b4c769e830251708ef10b.jpg', '3541f4b703934cb3a0d1ae51e559eb78.jpg', '477bbef38b8b4191939552b22ae9c5b4.jpg',

In [ ]:
df = pd.read_csv(CSV_PATH)
print(f'Total samples : {len(df)}')
print(f'Columns       : {df.columns.tolist()}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nClass balance: {df["label"].mean():.2%} AI / {1-df["label"].mean():.2%} Human')
df.head()

Total samples : 79950
Columns       : ['Unnamed: 0', 'file_name', 'label']

Label distribution:
label
1    39975
0    39975
Name: count, dtype: int64

Class balance: 50.00% AI / 50.00% Human


,Unnamed: 0,file_name,label
0,0,train_data/a6dcb93f596a43249135678dfcfc17ea.jpg,1
1,1,train_data/041be3153810433ab146bc97d5af505c.jpg,0
2,2,train_data/615df26ce9494e5db2f70e57ce7a3a4f.jpg,1
3,3,train_data/8542fe161d9147be8e835e50c0de39cd.jpg,0
4,4,train_data/5d81fa12bc3b4cea8c94a6700a477cf2.jpg,1


In [ ]:
@dataclass
class Config:
    # ── Data ──────────────────────────────────────────
    data_root   : str   = '/content/train_data'
    csv_path    : str   = folder_path + 'train.csv'
    train_size  : int   = 5_000      # None = Use all images
    val_ratio   : float = 0.20
    test_ratio  : float = 0.20
    num_workers : int   = 2

    # ── Model ─────────────────────────────────────────
    # Model architecture, e.g., 'resnet50', 'google/vit-base-patch16-224', 'microsoft/beit-base-patch16-224'
    model_name             : str  = 'resnet50'
    freeze_backbone        : bool = True
    unfreeze_last_n_blocks : int  = 0

    # ── Training ──────────────────────────────────────
    epochs          : int   = 20
    batch_size      : int   = 64
    lr              : float = 3e-4
    backbone_lr     : float = 1e-5
    weight_decay    : float = 1e-2
    label_smoothing : float = 0.1
    patience        : int   = 5
    grad_clip       : float = 1.0
    use_amp         : bool  = True

    # ── W&B ───────────────────────────────────────────
    project  : str  = 'vit-ai-detection'
    run_name : str  = 'vit-base-5k-head-only'
    tags     : list = field(default_factory=lambda: ['stage-1', 'head-only'])

    # ── Misc ──────────────────────────────────────────
    seed     : int = 42
    save_dir : str = '/content/drive/MyDrive/vit_checkpoints'

cfg = Config()
print(asdict(cfg))

{'data_root': '/content/train_data', 'csv_path': '/content/drive/MyDrive/NEU/DS5500-Data Capstone/DS5500_AIGI-Detection/AIGI-data/train.csv', 'train_size': 5000, 'val_ratio': 0.2, 'test_ratio': 0.2, 'num_workers': 2, 'model_name': 'resnet50', 'freeze_backbone': True, 'unfreeze_last_n_blocks': 0, 'epochs': 20, 'batch_size': 64, 'lr': 0.0003, 'backbone_lr': 1e-05, 'weight_decay': 0.01, 'label_smoothing': 0.1, 'patience': 5, 'grad_clip': 1.0, 'use_amp': True, 'project': 'vit-ai-detection', 'run_name': 'vit-base-5k-head-only', 'tags': ['stage-1', 'head-only'], 'seed': 42, 'save_dir': '/content/drive/MyDrive/vit_checkpoints'}


# Load data into Dataset

We do not have labels in test set -> do not use test_data_v2 at all!

## Prepare Data Splits

Filter the dataframe based on `cfg.train_size` and split it into training, validation, and test sets according to `cfg.val_ratio` and `cfg.test_ratio` for evaluation.


In [ ]:
if cfg.train_size is not None:
    df_sampled, _ = train_test_split(
    df,
    train_size=cfg.train_size,
    stratify=df["label"],
    random_state=cfg.seed)
else:
    df_sampled = df.copy()

# Calculate the sum of validation and test ratios for the first split
val_test_ratio_sum = cfg.val_ratio + cfg.test_ratio

# Split df_sampled into training and a temporary set (for validation and test)
df_train, df_temp = train_test_split(
    df_sampled,
    test_size=val_test_ratio_sum, # proportion of data for val+test
    stratify=df_sampled['label'],
    random_state=cfg.seed
)

# Calculate the test_size for the second split (df_temp into val and test)
# This is the proportion of the temporary set that should go to the test set
test_split_ratio_from_temp = cfg.test_ratio / val_test_ratio_sum

# Split df_temp into validation and test sets
df_val, df_test = train_test_split(
    df_temp,
    test_size=test_split_ratio_from_temp, # proportion of df_temp for test
    stratify=df_temp['label'],
    random_state=cfg.seed
)

print(f"Train set size: {len(df_train)}")
print(f"Validation set size: {len(df_val)}")
print(f"Test set size: {len(df_test)}")

Train set size: 3000
Validation set size: 1000
Test set size: 1000


## Define Image Transformations

Create `torchvision.transforms` for data augmentation (like random horizontal flip, color jitter) and preprocessing (resize, center crop, normalize) suitable for the model, applying different transformations for training and validation/test sets. We do this to make the task harder in training which forces the model to learn more robust features.


In [ ]:
import torchvision.transforms as transforms

# ImageNet mean and std (common for pre-trained models)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Training transformations with data augmentation
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224), # Randomly crop and resize to 224x224
    transforms.RandomHorizontalFlip(), # Randomly flip the image horizontally
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1), # Randomly change brightness, contrast, saturation, and hue
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) # Normalize with ImageNet mean and std
])

# Validation and Test transformations (preprocessing only, no augmentation)
val_test_transforms = transforms.Compose([
    transforms.Resize(256), # Resize the image to 256x256
    transforms.CenterCrop(224), # Crop the center of the image to 224x224
    transforms.ToTensor(), # Convert PIL Image to PyTorch Tensor
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD) # Normalize with ImageNet mean and std
])

print("Defined train_transforms and val_test_transforms.")

Defined train_transforms and val_test_transforms.


## Implement Custom Dataset

Create a `torch.utils.data.Dataset` class that handles loading images from the specified `cfg.data_root` using `PIL.Image`, applies the defined transformations, and returns the processed image and its corresponding label.


In [ ]:
class AIDataset(Dataset):
    def __init__(self, dataframe, data_root, transform=None):
        self.dataframe = dataframe
        self.data_root = data_root
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = self.dataframe.iloc[idx]['file_name']
        label = self.dataframe.iloc[idx]['label']

        # Construct full image path. Split 'train_data/' from the filename if present.
        # This ensures compatibility with both direct filenames and 'train_data/filename.jpg' paths
        if img_name.startswith('train_data/') or img_name.startswith('test_data_v2/'):
            img_name = os.path.basename(img_name)

        img_path = os.path.join(self.data_root, img_name)

        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.float32)

# Instantiate datasets
train_dataset = AIDataset(dataframe=df_train, data_root=image_root_path, transform=train_transforms)
val_dataset = AIDataset(dataframe=df_val, data_root=image_root_path, transform=val_test_transforms)
test_dataset = AIDataset(dataframe=df_test, data_root=image_root_path, transform=val_test_transforms)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Validation dataset size: {len(val_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 3000
Validation dataset size: 1000
Test dataset size: 1000


## Initialize DataLoaders

Create `torch.utils.data.DataLoader` instances for the training, validation, and test sets using `cfg.batch_size` and `cfg.num_workers` to efficiently load and batch the data for model training.


In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_dataloader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    num_workers=cfg.num_workers,
    pin_memory=True # Optimize data transfer to GPU
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    num_workers=cfg.num_workers,
    pin_memory=True
)

print(f"Number of batches in train_dataloader: {len(train_dataloader)}")
print(f"Number of batches in val_dataloader:   {len(val_dataloader)}")
print(f"Number of batches in test_dataloader:  {len(test_dataloader)}")

Number of batches in train_dataloader: 47
Number of batches in val_dataloader:   16
Number of batches in test_dataloader:  16


## Load and Modify ResNet Model

Load a pre-trained ResNet-50 model from `torchvision.models` (as `cfg.model_name` is 'resnet50'). Modify its final classification layer (`fc` layer) to output 2 classes for binary classification. Implement conditional logic to freeze earlier layers if `cfg.freeze_backbone` is true and unfreeze the last `cfg.unfreeze_last_n_blocks` of the ResNet backbone.


In [ ]:
import torchvision.models as models

# 1. Load a pre-trained ResNet-50 model
model = models.resnet50(pretrained=True)

# 2. Get the number of input features of the last fully connected layer
num_ftrs = model.fc.in_features

# 3. Replace the original fc layer with a new nn.Linear layer for 2 classes
model.fc = nn.Linear(num_ftrs, 2)

# 4. Implement conditional freezing logic
if cfg.freeze_backbone:
    print("Freezing backbone layers...")
    # Freeze all parameters initially
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze the new fc layer
    for param in model.fc.parameters():
        param.requires_grad = True

    # Unfreeze the last N blocks if specified
    if cfg.unfreeze_last_n_blocks > 0:
        print(f"Unfreezing last {cfg.unfreeze_last_n_blocks} backbone blocks...")
        # ResNet structure: conv1, bn1, relu, maxpool, layer1, layer2, layer3, layer4, avgpool, fc
        # Unfreeze layer4, layer3, etc.
        blocks_to_unfreeze = [model.layer4, model.layer3, model.layer2, model.layer1]
        for i in range(min(cfg.unfreeze_last_n_blocks, len(blocks_to_unfreeze))):
            target_layer = blocks_to_unfreeze[i]
            for param in target_layer.parameters():
                param.requires_grad = True

# 5. Move the modified model to the device
model = model.to(device)

# 6. Print the model architecture
print("\nModel architecture after modifications:")
print(model)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 174MB/s]


Freezing backbone layers...

Model architecture after modifications:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Seq

Define the loss function that is required for training the model.



In [ ]:
import torch.nn.functional as F

# Define loss function with label smoothing
# For binary classification (2 classes), CrossEntropyLoss is appropriate.
# If labels are 0 and 1, we can pass them directly. If the output of the model
# is logits, CrossEntropyLoss handles softmax internally.

def cross_entropy_loss_with_label_smoothing(outputs, targets, num_classes=2, label_smoothing=0.1):
    # Apply label smoothing
    # For binary classification, targets are 0 or 1.
    # We need to convert them to one-hot for label smoothing if using NLLLoss with log_softmax
    # Or directly adjust probabilities for CrossEntropyLoss which expects class indices as targets

    # Convert targets to one-hot encoding for label smoothing calculation
    # CrossEntropyLoss expects class indices, so this step is just for smoothing logic.
    # For actual CE loss, we pass original targets.
    if targets.dim() == 1:
        # Convert scalar labels to one-hot for smoothing calculation
        one_hot = F.one_hot(targets.long(), num_classes=num_classes).float()
    else:
        one_hot = targets.float()

    # Smooth labels
    smooth_labels = one_hot * (1.0 - label_smoothing) + label_smoothing / num_classes

    # CrossEntropyLoss expects raw logits and integer class labels (0 or 1 for binary)
    # If targets are float (e.g., from one-hot), it expects probabilities. Here they are 0/1.
    # So, we convert original targets to long to be used by F.cross_entropy directly for true labels.
    loss = F.cross_entropy(outputs, targets.long(), label_smoothing=label_smoothing)

    return loss

# Use the custom function or directly specify label_smoothing in F.cross_entropy
criterion = lambda outputs, targets: cross_entropy_loss_with_label_smoothing(
    outputs, targets, num_classes=2, label_smoothing=cfg.label_smoothing
)

print(f"Loss function defined with label smoothing: {cfg.label_smoothing}")

Loss function defined with label smoothing: 0.1


Define the optimizer and the learning rate scheduler. This will prepare the model for the training loop.



In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Separate parameters for different learning rates
# Parameters that require gradients will be trained
# Only the fc layer and potentially unfreezed backbone layers will have requires_grad=True
optimizer_params = [
    {'params': model.fc.parameters(), 'lr': cfg.lr} # Learning rate for the classification head
]

# Add backbone parameters if they require gradients and were not frozen
# We check if any parameter in the backbone layers requires grad
backbone_params = []
for name, param in model.named_parameters():
    if param.requires_grad and 'fc' not in name: # Exclude fc layer as it's already added
        backbone_params.append(param)

if backbone_params:
    optimizer_params.append({'params': backbone_params, 'lr': cfg.backbone_lr})


# Initialize the optimizer
optimizer = optim.AdamW(optimizer_params, weight_decay=cfg.weight_decay)

# Initialize the learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs)

print(f"Optimizer defined with learning rate for head: {cfg.lr}, and backbone: {cfg.backbone_lr} (if active).")
print(f"Scheduler defined with T_max: {cfg.epochs}.")

Optimizer defined with learning rate for head: 0.0003, and backbone: 1e-05 (if active).
Scheduler defined with T_max: 20.


## Implement Training and Validation Functions

Define Python functions for performing one epoch of training (like forward pass, loss calculation, backward pass, optimizer step, gradient clipping with `cfg.grad_clip`, and mixed precision training with `torch.cuda.amp.autocast` and `GradScaler` if `cfg.use_amp` is true) and one epoch of validation.


In [ ]:
import torch.nn.functional as F

def train_one_epoch(model, dataloader, criterion, optimizer, device, scaler, grad_clip, use_amp):
    model.train()
    total_loss = 0
    for batch_idx, (inputs, labels) in enumerate(dataloader):
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        with autocast(enabled=use_amp):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        if use_amp:
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer) # Unscale gradients before clipping
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device, use_amp):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            labels = labels.to(device)

            with autocast(enabled=use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            total_loss += loss.item()

            # Convert outputs to probabilities and predictions
            probs = F.softmax(outputs, dim=1)[:, 1] # Probability of class 1 (AI)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(dataloader)
    return avg_loss, np.array(all_preds), np.array(all_labels)

print("Defined train_one_epoch and validate_one_epoch functions.")

Defined train_one_epoch and validate_one_epoch functions.


## Execute Model Training

Run the main training loop for `cfg.epochs`, iterating through training and validation functions. Implement logic for tracking metrics, saving the best performing model (based on validation loss or accuracy) to `cfg.save_dir`, and applying early stopping if validation performance does not improve for `cfg.patience` epochs.


In [ ]:
import wandb
import os

# 1. Initialize wandb
wandb.init(
    project=cfg.project,
    name=cfg.run_name,
    tags=cfg.tags,
    config=asdict(cfg)
)

# Create save directory if it doesn't exist
os.makedirs(cfg.save_dir, exist_ok=True)

# 2. Create an instance of torch.cuda.amp.GradScaler
scaler = GradScaler() if cfg.use_amp else None

# 3. Initialize variables for tracking best model and early stopping
best_val_loss = float('inf')
patience_counter = 0

print("Starting training...")
for epoch in range(1, cfg.epochs + 1):
    print(f"Epoch {epoch}/{cfg.epochs}")

    # 5. Training step
    train_loss = train_one_epoch(
        model, train_dataloader, criterion, optimizer, device, scaler, cfg.grad_clip, cfg.use_amp
    )

    # 6. Validation step
    val_loss, val_preds, val_labels = validate_one_epoch(
        model, val_dataloader, criterion, device, cfg.use_amp
    )

    # 7. Calculate metrics for the validation set
    val_roc_auc = roc_auc_score(val_labels, val_preds)

    # Binarize predictions for classification report and confusion matrix
    val_bin_preds = (val_preds > 0.5).astype(int)
    val_report = classification_report(val_labels, val_bin_preds, output_dict=True, zero_division=0)
    val_confusion_matrix = confusion_matrix(val_labels, val_bin_preds)

    # Extract key metrics from classification report
    val_accuracy = val_report['accuracy']
    val_precision = val_report['macro avg']['precision']
    val_recall = val_report['macro avg']['recall']
    val_f1 = val_report['macro avg']['f1-score']

    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss: {val_loss:.4f}")
    print(f"  Val ROC AUC: {val_roc_auc:.4f}")
    print(f"  Val Accuracy: {val_accuracy:.4f}")
    print(f"  Val Precision: {val_precision:.4f}")
    print(f"  Val Recall: {val_recall:.4f}")
    print(f"  Val F1-Score: {val_f1:.4f}")
    print(f"  Confusion Matrix:\n{val_confusion_matrix}")

    # 8. Log metrics to wandb
    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_roc_auc": val_roc_auc,
        "val_accuracy": val_accuracy,
        "val_precision": val_precision,
        "val_recall": val_recall,
        "val_f1_score": val_f1,
        "val_confusion_matrix": wandb.Table(data=val_confusion_matrix.tolist(), columns=["Pred 0", "Pred 1"])
    })

    # 9. Step the learning rate scheduler
    scheduler.step()

    # 10. Implement logic to save the best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        model_save_path = os.path.join(cfg.save_dir, 'best_model.pth')
        torch.save(model.state_dict(), model_save_path)
        print(f"  Saved best model to {model_save_path} (Val Loss: {best_val_loss:.4f})")
        wandb.run.summary["best_val_loss"] = best_val_loss
        wandb.run.summary["best_val_roc_auc"] = val_roc_auc
        wandb.run.summary["best_val_accuracy"] = val_accuracy
    else:
        patience_counter += 1
        print(f"  Validation loss did not improve. Patience: {patience_counter}/{cfg.patience}")

    # 11. Implement early stopping
    if patience_counter > cfg.patience:
        print(f"  Early stopping triggered after {cfg.patience} epochs without improvement.")
        break

print("Training finished.")

# 12. Finalize the wandb run
wandb.finish()

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 1


wandb: You chose 'Create a W&B account'
wandb: Create an account here: https://wandb.ai/authorize?signup=true&ref=models
wandb: After creating your account, create a new API key and store it securely.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: yaj001513 (yaj001513-northeastern-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


...

/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.3632
  Val Loss: 0.3457
  Val ROC AUC: 0.9748
  Val Accuracy: 0.9220
  Val Precision: 0.9221
  Val Recall: 0.9219
  Val F1-Score: 0.9220
  Confusion Matrix:
[[469  35]
 [ 43 453]]
  Validation loss did not improve. Patience: 3/5
Epoch 20/20


/tmp/ipython-input-1765662441.py:12: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):
/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Train Loss: 0.3648
  Val Loss: 0.3462
  Val ROC AUC: 0.9749
  Val Accuracy: 0.9230
  Val Precision: 0.9233
  Val Recall: 0.9229
  Val F1-Score: 0.9230
  Confusion Matrix:
[[471  33]
 [ 44 452]]
  Validation loss did not improve. Patience: 4/5
Training finished.


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_loss,█▄▃▂▂▂▂▂▁▂▁▂▁▁▁▂▁▁▁▁
val_accuracy,▁▅▆▆▆▇▇▇██▇█████████
val_f1_score,▁▅▆▆▆▇▇▇██▇█████████
val_loss,█▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
val_precision,▁▅▆▆▆▇▇▇▇▇▇█████████
val_recall,▁▅▆▆▆▇▇▇██▇█████████
val_roc_auc,▁▄▅▆▇▇▇▇▇▇██████████
best_val_accuracy,0.923
best_val_loss,0.34439
best_val_roc_auc,0.97486


Evaluate the best performing model on the unseen test set to assess its generalization capabilities.



In [ ]:
print("\n--- Evaluating best model on Test Set ---")

# Re-initialize wandb for test evaluation if needed (training run might have ended)
# We'll create a new run specific for logging test results, linking it to the previous run if desired.
# For simplicity, starting a fresh run for test results here.
import wandb
import os

wandb.init(
    project=cfg.project, # Use the same project
    name=f"{cfg.run_name}-test-evaluation", # A new run name for test evaluation
    tags=cfg.tags + ["test-evaluation"],
    config=asdict(cfg) # Log the configuration again
)

# 1. Load the best model
model.load_state_dict(torch.load(os.path.join(cfg.save_dir, 'best_model.pth')))

# 2. Run validation on the test set
test_loss, test_preds, test_labels = validate_one_epoch(
    model, test_dataloader, criterion, device, cfg.use_amp
)

# 3. Calculate metrics for the test set
test_roc_auc = roc_auc_score(test_labels, test_preds)

# Binarize predictions for classification report and confusion matrix
test_bin_preds = (test_preds > 0.5).astype(int)
test_report = classification_report(test_labels, test_bin_preds, output_dict=True, zero_division=0)
test_confusion_matrix = confusion_matrix(test_labels, test_bin_preds)

# Extract key metrics from classification report
test_accuracy = test_report['accuracy']
test_precision = test_report['macro avg']['precision']
test_recall = test_report['macro avg']['recall']
test_f1 = test_report['macro avg']['f1-score']

print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test ROC AUC: {test_roc_auc:.4f}")
print(f"  Test Accuracy: {test_accuracy:.4f}")
print(f"  Test Precision: {test_precision:.4f}")
print(f"  Test Recall: {test_recall:.4f}")
print(f"  Test F1-Score: {test_f1:.4f}")
print(f"  Test Confusion Matrix:\n{test_confusion_matrix}")

# 4. Log test metrics to wandb
wandb.log({
    "test_loss": test_loss,
    "test_roc_auc": test_roc_auc,
    "test_accuracy": test_accuracy,
    "test_precision": test_precision,
    "test_recall": test_recall,
    "test_f1_score": test_f1,
    "test_confusion_matrix": wandb.Table(data=test_confusion_matrix.tolist(), columns=["Pred 0", "Pred 1"])
})

print("Test set evaluation complete and metrics logged to Weights & Biases.")

# Finalize the wandb run for test evaluation
wandb.finish()


--- Evaluating best model on Test Set ---


/tmp/ipython-input-1765662441.py:43: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


  Test Loss: 0.3655
  Test ROC AUC: 0.9662
  Test Accuracy: 0.9020
  Test Precision: 0.9020
  Test Recall: 0.9020
  Test F1-Score: 0.9020
  Test Confusion Matrix:
[[455  50]
 [ 48 447]]
Test set evaluation complete and metrics logged to Weights & Biases.


test_accuracy,▁
test_f1_score,▁
test_loss,▁
test_precision,▁
test_recall,▁
test_roc_auc,▁
test_accuracy,0.902
test_f1_score,0.90199
test_loss,0.36546
test_precision,0.90198
test_recall,0.90201
